# Smart Grid Energy Forecasting

This notebook contains the complete daily forecasting workflow used by the project: loading the prepared data, creating time and lag features, training chronological XGBoost models, evaluating them, saving the artifacts, and predicting the grid balance for a sample input.

The lag model is the main model. The fallback model can be used when the previous realized loads are unavailable.

## 1. Imports and paths

Run the notebook from the project directory. Install the packages in `requirements.txt` first if the imports are unavailable.

In [ ]:
import datetime as dt
import json
import math
import pickle
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

ROOT = Path.cwd()
if not (ROOT / 'combined.csv').exists():
    raise FileNotFoundError('Open this notebook from the Smart Grid Energy Forecasting project directory.')

DATA_PATH = ROOT / 'combined.csv'
WEATHER_PATH = ROOT / 'weather_data.csv'
ARTIFACT_DIR = ROOT / 'artifacts_daily'
MODEL_PATH = ARTIFACT_DIR / 'daily_xgboost_model.pkl'
LAGS = (1, 7)
LATITUDE = 50.1109
LONGITUDE = 8.6821

print(f'Project directory: {ROOT}')

## 2. Load and inspect the daily data

Each row contains one UTC date, average generation, realized load, and mean temperature.

In [ ]:
data = pd.read_csv(DATA_PATH, parse_dates=['date'])
data['date'] = pd.to_datetime(data['date'], utc=True)

required_columns = {
    'date', 'current_energy_generation', 'realized_load', 'temperature_celsius'
}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')
if data[list(required_columns)].isna().any().any():
    raise ValueError('The training data contains missing values.')

print(f'Rows: {len(data):,}')
print(f'Date range: {data.date.min().date()} to {data.date.max().date()}')
display(data.head())

## 3. Feature engineering and model definition

The sine and cosine fields represent annual seasonality without an artificial discontinuity between December 31 and January 1. The main model additionally uses realized load from one and seven days earlier.

In [ ]:
def base_features(frame):
    date = frame['date']
    return pd.DataFrame({
        'current_energy_generation': frame['current_energy_generation'],
        'temperature_celsius': frame['temperature_celsius'],
        'day_of_week': date.dt.dayofweek,
        'day_of_year_sin': np.sin(2 * np.pi * date.dt.dayofyear / 365.25),
        'day_of_year_cos': np.cos(2 * np.pi * date.dt.dayofyear / 365.25),
    })


def make_model():
    return XGBRegressor(
        n_estimators=1500,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.85,
        colsample_bytree=0.90,
        min_child_weight=5,
        reg_lambda=4,
        objective='reg:squarederror',
        eval_metric='rmse',
        early_stopping_rounds=75,
        tree_method='hist',
        n_jobs=-1,
        random_state=42,
    )


def train_model(features, target, fit_end, test_start):
    model = make_model()
    model.fit(
        features.iloc[:fit_end],
        target.iloc[:fit_end],
        eval_set=[(features.iloc[fit_end:test_start], target.iloc[fit_end:test_start])],
        verbose=False,
    )
    prediction = model.predict(features.iloc[test_start:])
    actual = target.iloc[test_start:]
    metrics = {
        'r2': float(r2_score(actual, prediction)),
        'mae': float(mean_absolute_error(actual, prediction)),
        'rmse': float(mean_squared_error(actual, prediction) ** 0.5),
        'best_iteration': int(model.best_iteration),
    }
    return model, prediction, metrics

## 4. Train, evaluate, and save both models

The split is chronological: the first 72% is used for fitting, the next 8% for early stopping, and the final 20% for testing. Running this cell replaces the saved model artifacts with newly trained ones.

In [ ]:
ARTIFACT_DIR.mkdir(exist_ok=True)
target = data['realized_load']
test_start = int(len(data) * 0.80)
fit_end = int(test_start * 0.90)

fallback_model, fallback_prediction, fallback_metrics = train_model(
    base_features(data), target, fit_end, test_start
)

lagged_features = base_features(data)
for lag in LAGS:
    lagged_features[f'load_lag_{lag}'] = target.shift(lag)

valid_start = max(LAGS)
lag_model, lag_prediction, lag_metrics = train_model(
    lagged_features.iloc[valid_start:].reset_index(drop=True),
    target.iloc[valid_start:].reset_index(drop=True),
    fit_end - valid_start,
    test_start - valid_start,
)

models = {'lag': lag_model, 'fallback': fallback_model}
with MODEL_PATH.open('wb') as file:
    pickle.dump(models, file)

actual = target.iloc[test_start:]
test_predictions = pd.DataFrame({
    'date': data['date'].iloc[test_start:].to_numpy(),
    'actual_realized_load': actual.to_numpy(),
    'predicted_realized_load': lag_prediction,
    'residual': actual.to_numpy() - lag_prediction,
})
test_predictions.to_csv(ARTIFACT_DIR / 'test_predictions.csv', index=False)

metrics = {
    **lag_metrics,
    'daily_rows': len(data),
    'test_rows': len(actual),
    'test_period': [
        data['date'].iloc[test_start].isoformat(),
        data['date'].iloc[-1].isoformat(),
    ],
    'features': list(lagged_features.columns),
    'lags': list(LAGS),
    'fallback_r2': fallback_metrics['r2'],
}
(ARTIFACT_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2) + '\n')

display(pd.DataFrame([
    {'model': 'Lag model', **lag_metrics},
    {'model': 'Fallback model', **fallback_metrics},
]).set_index('model'))

## 5. Prediction helpers

Temperature is read from the local weather file when possible. For a date outside that file, `fetch_temperature` uses Open-Meteo and therefore needs internet access. A temperature may always be supplied directly instead.

In [ ]:
def parse_date(date_text):
    try:
        return dt.date.fromisoformat(date_text)
    except (TypeError, ValueError) as error:
        raise ValueError('Enter the date in YYYY-MM-DD format.') from error


def validate_inputs(date_text, generation, temperature):
    date = parse_date(date_text)
    if not math.isfinite(generation) or generation < 0:
        raise ValueError('Current energy generation must be a non-negative number.')
    if not math.isfinite(temperature):
        raise ValueError('Temperature must be a valid number.')
    return date


def build_features(date_text, generation, temperature):
    date = validate_inputs(date_text, generation, temperature)
    timestamp = pd.Timestamp(date)
    return pd.DataFrame([{
        'current_energy_generation': float(generation),
        'temperature_celsius': float(temperature),
        'day_of_week': timestamp.dayofweek,
        'day_of_year_sin': np.sin(2 * np.pi * timestamp.dayofyear / 365.25),
        'day_of_year_cos': np.cos(2 * np.pi * timestamp.dayofyear / 365.25),
    }])


def fetch_temperature(date_text):
    date = parse_date(date_text)
    weather = pd.read_csv(WEATHER_PATH, parse_dates=['date'])
    weather['date'] = pd.to_datetime(weather['date'], utc=True).dt.date
    saved = weather.set_index('date')['temperature_celsius'].get(date)
    if saved is not None and not pd.isna(saved):
        return float(saved)

    if date > dt.date.today() + dt.timedelta(days=16):
        raise ValueError(
            'Temperature forecasts are available only up to 16 days ahead. '
            'Enter the expected temperature manually for this date.'
        )

    historical = date < dt.date.today() - dt.timedelta(days=5)
    endpoint = (
        'https://archive-api.open-meteo.com/v1/archive'
        if historical else 'https://api.open-meteo.com/v1/forecast'
    )
    query = urlencode({
        'latitude': LATITUDE,
        'longitude': LONGITUDE,
        'start_date': date.isoformat(),
        'end_date': date.isoformat(),
        'daily': 'temperature_2m_mean',
        'timezone': 'UTC',
    })
    try:
        with urlopen(f'{endpoint}?{query}', timeout=15) as response:
            result = json.load(response)
        return float(result['daily']['temperature_2m_mean'][0])
    except (HTTPError, URLError, OSError, json.JSONDecodeError, KeyError,
            IndexError, TypeError, ValueError) as error:
        raise ValueError(f'Temperature is unavailable for {date.isoformat()}.') from error


def load_saved_models(path=MODEL_PATH):
    if not path.exists():
        raise FileNotFoundError('Run the training cell before making a prediction.')
    with path.open('rb') as file:
        return pickle.load(file)


def predict_realized_load(date_text, generation, temperature, models,
                          use_lags=True, load_lag_1=None, load_lag_7=None):
    features = build_features(date_text, generation, temperature)
    if not use_lags:
        return float(models['fallback'].predict(features)[0])

    lags = (load_lag_1, load_lag_7)
    if any(value is None or not math.isfinite(value) or value < 0 for value in lags):
        raise ValueError(
            'Enter valid non-negative realized load values for yesterday and seven days ago.'
        )
    features['load_lag_1'] = float(load_lag_1)
    features['load_lag_7'] = float(load_lag_7)
    return float(models['lag'].predict(features)[0])


def calculate_balance(generation, predicted_load):
    if not math.isfinite(predicted_load) or predicted_load <= 0:
        raise ValueError('Predicted realized load must be greater than zero.')
    difference = float(generation - predicted_load)
    percentage = difference / predicted_load * 100
    if abs(percentage) <= 5:
        status = 'Balanced'
    elif difference > 0:
        status = 'Overproducing'
    else:
        status = 'Underproducing'
    return {'status': status, 'difference': difference, 'percentage': percentage}

## 6. Sample input and prediction

The sample uses an actual row from January 15, 2024. Its lag inputs are the realized loads from January 14 and January 8. Change the values in `sample_input` to make another prediction.

In [ ]:
sample_input = {
    'date': '2024-01-15',
    'current_energy_generation': 15774.934896,
    'temperature_celsius': 0.229167,
    'use_lags': True,
    'load_lag_1': 13605.825521,
    'load_lag_7': 15682.007812,
}

models = load_saved_models()
predicted_load = predict_realized_load(
    sample_input['date'],
    sample_input['current_energy_generation'],
    sample_input['temperature_celsius'],
    models,
    use_lags=sample_input['use_lags'],
    load_lag_1=sample_input['load_lag_1'],
    load_lag_7=sample_input['load_lag_7'],
)
balance = calculate_balance(sample_input['current_energy_generation'], predicted_load)

sample_result = pd.DataFrame([{
    'date': sample_input['date'],
    'generation': sample_input['current_energy_generation'],
    'predicted_load': predicted_load,
    'difference': balance['difference'],
    'difference_percent': balance['percentage'],
    'status': balance['status'],
}])
display(sample_result.round({
    'generation': 2,
    'predicted_load': 2,
    'difference': 2,
    'difference_percent': 2,
}))

### Fallback example

Set `use_lags=False` when previous realized-load values are not known.

In [ ]:
fallback_prediction = predict_realized_load(
    date_text='2030-01-15',
    generation=14000.0,
    temperature=5.5,
    models=models,
    use_lags=False,
)
fallback_balance = calculate_balance(14000.0, fallback_prediction)
print(f'Predicted load: {fallback_prediction:,.2f}')
print(f"Status: {fallback_balance['status']} "
      f"({fallback_balance['percentage']:+.2f}%)")